In [ ]:
!pip install pdfplumber

In [ ]:
import pdfplumber
import json
import re
from pathlib import Path

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_pages.append(page_text)
    return "\n".join(text_pages)

In [ ]:
def is_chapter_title(line: str) -> bool:
    line = line.strip()
    if len(line) < 5 or len(line) > 80:
        return False
    if sum(c.isupper() for c in line) / max(len(line), 1) > 0.6:
        return True
    return False

In [ ]:
def structure_text(text: str) -> dict:
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    chapters = []
    current_chapter = {
        "id": 0,
        "title": "Chapitre 1",
        "paragraphs": []
    }

    chapter_id = 0
    buffer = []

    def flush_paragraph():
        nonlocal buffer
        if buffer:
            paragraph = " ".join(buffer).strip()
            if len(paragraph.split()) >= 5:
                current_chapter["paragraphs"].append(paragraph)
            buffer = []

    for line in lines:
        if is_chapter_title(line):
            flush_paragraph()
            if current_chapter["paragraphs"]:
                chapters.append(current_chapter)

            chapter_id += 1
            current_chapter = {
                "id": chapter_id,
                "title": line,
                "paragraphs": []
            }
        else:
            if line.endswith(".") or line.endswith("!") or line.endswith("?"):
                buffer.append(line)
                flush_paragraph()
            else:
                buffer.append(line)

    flush_paragraph()
    if current_chapter["paragraphs"]:
        chapters.append(current_chapter)

    return chapters

In [ ]:
def pdf_to_structured_json(pdf_path: str, output_json: str):
    raw_text = extract_text_from_pdf(pdf_path)
    chapters = structure_text(raw_text)

    data = {
        "source": Path(pdf_path).name,
        "chapters": chapters
    }

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"JSON enregistré : {output_json}")

In [ ]:
pdf_to_structured_json(
    "/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/Sarkozy journal d'un prisonnier.pdf",
    "/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/livres_sarkozy/journal_prisonnier_sarkozy.json"
)

## nettoyage car scories océrisation

In [ ]:
PATTERN = re.compile(
    r"446699[\w\d]*.*?[\w\d]*5588",
    flags=re.DOTALL
)

def clean_paragraph(text: str) -> str:
    if not text:
        return text
    cleaned = PATTERN.sub("", text)
    return cleaned.strip()

In [ ]:
input_path = Path(
    "/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/livres_sarkozy/journal_prisonnier_sarkozy_clean_clean.json"
)

output_path = input_path.with_name(
    input_path.stem + "_clean.json"
)

with open(input_path, encoding="utf-8") as f:
    data = json.load(f)

for chapter in data.get("chapters", []):
    new_paragraphs = []
    for p in chapter.get("paragraphs", []):
        p_clean = clean_paragraph(p)
        if p_clean:
            new_paragraphs.append(p_clean)
    chapter["paragraphs"] = new_paragraphs

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

output_path

## suppression numéros de page

In [ ]:
# paragraphe = uniquement un numéro (page)
PAGE_ONLY = re.compile(r"^\s*\d{1,3}\s*$")

# numéro de page en fin de paragraphe (après espace / tiret)
PAGE_AT_END = re.compile(r"[\s\-–—]+\d{1,3}\s*$")

def clean_paragraph_page_numbers(text: str) -> str:
    if not text:
        return ""

    # Cas 1 : paragraphe = numéro seul
    if PAGE_ONLY.match(text):
        return ""

    # Cas 2 : numéro de page en fin de paragraphe
    text = PAGE_AT_END.sub("", text)

    return text.strip()

In [ ]:
input_path = Path(
    "/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/livres_sarkozy/journal_prisonnier_sarkozy.json"
)

output_path = input_path.with_name(
    input_path.stem.replace("_clean", "") + "_nopages.json"
)

with open(input_path, encoding="utf-8") as f:
    data = json.load(f)

for chapter in data.get("chapters", []):
    cleaned = []
    for p in chapter.get("paragraphs", []):
        p2 = clean_paragraph_page_numbers(p)
        if p2:
            cleaned.append(p2)
    chapter["paragraphs"] = cleaned

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

output_path